In [2]:
from keras.models import Sequential
from keras.layers import Conv2D, ZeroPadding2D, Activation, Input, concatenate, Layer
from keras.models import Model
from keras.layers import BatchNormalization, MaxPooling2D, AveragePooling2D, Concatenate, Lambda, Flatten, Dense
from keras.initializers import glorot_uniform
from keras import backend as K

K.set_image_data_format('channels_last')
# import cv2
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from fr_utils import *
from inception_blocks_v2 import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Encoding face images into a 128-dimensional vector

In [4]:
FRmodel = faceRecoModel(input_shape=(96, 96, 3))

In [5]:
print("Total Params:", FRmodel.count_params())

Total Params: 3743280


Triplet Loss

In [6]:
def triplet_loss(y_true, y_pred, alpha=0.2):
    """
    Implementation of the triplet loss as defined by the triplet loss formula

    Arguments:
    y_true -- true labels, required when you define a loss in keras, you don't need it in this function
    y_pred -- python list containing three objects:
            anchor -- the encodings for the anchor images, of shaoe (None, 128)
            positive -- the encodings for the positive images, of shape (None, 128)
            negative -- the encodings for the negative images, of shape (None, 128)
    Returns:
    loss -- real number, value of the loss
    """

    anchor, positive, negative = y_pred[0], y_pred[1], y_pred[2]

    pos_dist = tf.reduce_sum(tf.square(anchor, positive), axis=-1)

    neg_dist = tf.reduce_sum(tf.square(anchor, negative), axis=-1)

    basic_loss = pos_dist - neg_dist + alpha

    loss = tf.reduce_sum(tf.maximum(basic_loss, 0.0))

    return loss

In [7]:
y_true = (None, None, None)
y_pred = (tf.random.normal([3, 128], mean=6, stddev=0.1, seed=1),
          tf.random.normal([3, 128], mean=1, stddev=1, seed=1),
          tf.random.normal([3, 128], mean=3, stddev=4, seed=1))

loss = triplet_loss(y_true, y_pred)

print('loss = ' + str(loss.numpy()))

loss = 0.6


Loading the pre-trained model

In [8]:
FRmodel.compile(optimizer='adam', loss=triplet_loss, metrics=['accuracy'])
load_weights_from_FaceNet(FRmodel)

# Applying the Model

Face Verification

In [9]:
database = {}
database["danielle"] = img_to_encoding("images/danielle.png", FRmodel)
database["younes"] = img_to_encoding("images/younes.jpg", FRmodel)
database["tian"] = img_to_encoding("images/tian.jpg", FRmodel)
database["andrew"] = img_to_encoding("images/andrew.jpg", FRmodel)
database["kian"] = img_to_encoding("images/kian.jpg", FRmodel)
database["dan"] = img_to_encoding("images/dan.jpg", FRmodel)
database["sebastiano"] = img_to_encoding("images/sebastiano.jpg", FRmodel)
database["bertrand"] = img_to_encoding("images/bertrand.jpg", FRmodel)
database["kevin"] = img_to_encoding("images/kevin.jpg", FRmodel)
database["felix"] = img_to_encoding("images/felix.jpg", FRmodel)
database["benoit"] = img_to_encoding("images/benoit.jpg", FRmodel)
database["arnaud"] = img_to_encoding("images/arnaud.jpg", FRmodel)

In [10]:
def verify(image_path, identity, database, model):
    """
    Function that verifies if the person on the "image_path" image is "identity".

    Arguments:
    image_path -- path to an image
    identity -- string, name of the person you'd like to verify the identity. Has to be an employee who works in the office.
    database -- python dictionary mapping names of allowed people's names (strings) to their encodings (vectors).
    model -- your inception model instance in keras

    Returns:
    dist -- distance between the image_path and the image of "identity" in the database.
    door_open -- True, if the door should open. False otherwise
    """
    encoding = img_to_encoding(image_path, model)

    dist = np.linalg.norm(encoding - database[identity])

    if dist < 0.7:
        print("It's " + str(identity) + ", welcome in!")
        door_open = True
    else:
        print("It's not " + str(identity) + ", please go away")
        door_open = False

    return dist, door_open

In [11]:
verify("images/camera_0.jpg", "younes", database, FRmodel)

It's younes, welcome in!


(np.float32(0.66714036), True)

In [13]:
verify("images/camera_2.jpg", "kian", database, FRmodel)

It's not kian, please go away


(np.float32(0.8586885), False)

Face Recognition

In [16]:
def who_is_it(image_path, database, model):
    """
    Implements face recognition for the office by finding who is the person on the image_path image.

    Arguments:
    image_path -- path to an image
    database -- database containing image encodings along with the name of the person on the image
    model -- your Inception model instance in keras

    Returns:
    min_dist -- the minimum distance between image_path encoding and the encodings from the database
    identity -- string, the name prediction for the person on image_path
    """

    encoding = img_to_encoding(image_path, model)

    min_dist = 100

    for (name, db_enc) in database.items():

        dist = np.linalg.norm(encoding - db_enc)

        if dist < min_dist:

            min_dist = dist
            identity = name

    if min_dist > 0.7:
        print("Not in the database.")
    else:
        print("It's " + str(identity) + ", the distance is " + str(min_dist))

    return min_dist, identity

In [18]:
who_is_it("images/camera_1.jpg", database, FRmodel)

It's bertrand, the distance is 0.46807343


(np.float32(0.46807343), 'bertrand')